# `ModelCallLimitMiddleware`

Middleware that tracks model-call counts and prevents an agent from making more model calls than the configured limits allow.

It supports two independent limits:

- **Thread limit** — Persists the model-call count across multiple runs of the same agent thread.
- **Run limit** — Counts model calls only within the current agent run.

When a limit is reached, the middleware can either end agent execution gracefully or raise an exception.

- Bases: `AgentMiddleware[ModelCallLimitState[ResponseT], ContextT, ResponseT]`

## Constructor

```python
ModelCallLimitMiddleware(
    *,
    thread_limit: int | None = None, # Maximum calls across the thread
    run_limit: int | None = None, # Maximum calls during one run
    exit_behavior: Literal["end", "error"] = "end" # Limit handling behaviour
) -> None
```

## Parameters

* `thread_limit` — Maximum number of model calls allowed across the agent thread.
  * The count persists across multiple invocations when thread state is persisted.
  * `None` disables the thread-level limit.
* `run_limit` — Maximum number of model calls allowed during a single agent invocation.
  * The count is not tracked across separate runs.
  * `None` disables the run-level limit.
* `exit_behavior` — Determines what happens when either configured limit has been reached.
  * `"end"` — Jumps to the end of agent execution and adds an artificial `AIMessage` explaining which limit was exceeded.
  * `"error"` — Raises `ModelCallLimitExceededError`.

At least one of `thread_limit` or `run_limit` must be provided.

## Attributes

* `thread_limit` — Stores the configured thread-level model-call limit.
* `run_limit` — Stores the configured run-level model-call limit.
* `exit_behavior` — Stores the selected limit-handling strategy.
* `state_schema` — Uses `ModelCallLimitState` as the middleware state schema.

## Methods

### 1. `before_model`

Checks the current call counts before another model call is made.

- **Syntax:**

```python
before_model(
    self,
    state: ModelCallLimitState[ResponseT], # Current agent state and counters
    runtime: Runtime[ContextT] # Current LangGraph runtime
) -> dict[str, Any] | None
```

- **Behaviour:**
  * Reads `thread_model_call_count` and `run_model_call_count` from the state.
  * Uses `0` when either counter is absent.
  * Checks whether a configured limit has already been reached.
  * Allows the model call when neither limit has been reached.

- **Returns:**
  * `None` when the next model call is allowed.
  * When `exit_behavior="end"`, returns:

```python
{
    "jump_to": "end",
    "messages": [AIMessage(content="Model call limits exceeded: ...")]
}
```

- **Raises:**
  * `ModelCallLimitExceededError` — When a limit is reached and `exit_behavior="error"`.

The hook is configured with `can_jump_to=["end"]`.

### 2. `abefore_model`

Asynchronous version of `before_model`.

- **Syntax:**

```python
async abefore_model(
    self,
    state: ModelCallLimitState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

- It delegates directly to `before_model`.
- It has the same return values, exception behaviour, and `can_jump_to=["end"]` hook configuration.

### 3. `after_model`

Increments the thread-level and run-level counters after a model call completes.

- **Syntax:**

```python
after_model(
    self,
    state: ModelCallLimitState[ResponseT], # Current agent state
    runtime: Runtime[ContextT] # Current LangGraph runtime
) -> dict[str, Any] | None
```

- **Returns:**

```python
{
    "thread_model_call_count": current_thread_count + 1,
    "run_model_call_count": current_run_count + 1
}
```

Missing counters are treated as `0` before incrementing.

### 4. `aafter_model`

Asynchronous version of `after_model`.

- **Syntax:**

```python
async aafter_model(
    self,
    state: ModelCallLimitState[ResponseT],
    runtime: Runtime[ContextT]
) -> dict[str, Any] | None
```

- It delegates directly to `after_model`.

---

# Supporting Types

## `ModelCallLimitState`

State schema used by `ModelCallLimitMiddleware` to store model-call counters.

- Bases: `AgentState[ResponseT]`

```python
class ModelCallLimitState(AgentState[ResponseT]):
    thread_model_call_count: NotRequired[
        Annotated[int, PrivateStateAttr]
    ]
    run_model_call_count: NotRequired[
        Annotated[int, UntrackedValue, PrivateStateAttr]
    ]
```

## Fields

* `thread_model_call_count` — Number of model calls made in the current thread.
  * Optional state field.
  * Marked as a private state attribute.
  * Can persist across multiple runs when the agent thread is checkpointed.
* `run_model_call_count` — Number of model calls made during the current run.
  * Optional state field.
  * Marked as both `UntrackedValue` and a private state attribute.
  * It is not accumulated through the normal tracked channel mechanism across runs.

## Type parameter

* `ResponseT` — Type of the structured response stored by the inherited agent state. It defaults to `Any` through the surrounding middleware generics.

---

# `ModelCallLimitExceededError`

Exception raised when a model-call limit is reached and the middleware uses `exit_behavior="error"`.

- Bases: `Exception`

## Constructor

```python
ModelCallLimitExceededError(
    thread_count: int, # Current thread-level count
    run_count: int, # Current run-level count
    thread_limit: int | None, # Configured thread limit
    run_limit: int | None # Configured run limit
) -> None
```

## Attributes

* `thread_count` — Current number of model calls recorded for the thread.
* `run_count` — Current number of model calls recorded for the run.
* `thread_limit` — Configured thread-level limit, or `None` when disabled.
* `run_limit` — Configured run-level limit, or `None` when disabled.

The exception message identifies every limit that has been reached. For example:

```text
Model call limits exceeded: thread limit (10/10), run limit (5/5)
```

---

# Internal Helper

## `_build_limit_exceeded_message`

Builds the human-readable message used by both the graceful ending behaviour and `ModelCallLimitExceededError`.

> This is an internal module-level helper function.

- **Syntax:**

```python
_build_limit_exceeded_message(
    thread_count: int,
    run_count: int,
    thread_limit: int | None,
    run_limit: int | None
) -> str
```

- Adds the thread limit to the message when `thread_count >= thread_limit`.
- Adds the run limit to the message when `run_count >= run_limit`.
- Includes both descriptions when both limits have been reached.

---

# Limit Behaviour

The limit is checked **before** a new model call and the counters are incremented **after** each completed model call.

For example, with `run_limit=3`:

1. The first model call is allowed; the run count becomes `1` afterward.
2. The second model call is allowed; the run count becomes `2` afterward.
3. The third model call is allowed; the run count becomes `3` afterward.
4. Before a fourth model call, the middleware detects `3 >= 3` and applies `exit_behavior`.

When both limits are configured, reaching either one is sufficient to stop execution or raise the exception.

---

# Exceptions

The `ModelCallLimitMiddleware` constructor raises `ValueError` in these cases:

```python
# Neither limit was supplied
ModelCallLimitMiddleware()
```

```text
At least one limit must be specified (thread_limit or run_limit)
```

```python
# Unsupported exit behaviour
ModelCallLimitMiddleware(
    run_limit=5,
    exit_behavior="stop" # type: ignore[arg-type]
)
```

```text
Invalid exit_behavior: stop. Must be 'end' or 'error'
```

---

# Usage Examples

## End the agent gracefully

```python
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

call_limit = ModelCallLimitMiddleware(
    thread_limit=10,
    run_limit=5,
    exit_behavior="end",
)

agent = create_agent(
    model="openai:gpt-5.5",
    tools=[],
    middleware=[call_limit],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Complete this task."}]},
    config={"configurable": {"thread_id": "example-thread"}},
)
```

Once a configured limit is reached, the returned state contains an artificial `AIMessage` describing the exceeded limit and execution jumps to the end.

## Raise an exception

```python
from langchain.agents.middleware import (
    ModelCallLimitExceededError,
    ModelCallLimitMiddleware,
)

call_limit = ModelCallLimitMiddleware(
    run_limit=3,
    exit_behavior="error",
)

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "Complete this task."}]}
    )
except ModelCallLimitExceededError as error:
    print(error.run_count)
    print(error.run_limit)
    print(str(error))
```

## Apply only a thread-level limit

```python
call_limit = ModelCallLimitMiddleware(thread_limit=20)
```

## Apply only a run-level limit

```python
call_limit = ModelCallLimitMiddleware(run_limit=6)
```